# Multilingual narrative and propaganda analysis

The central model combines a cross-lingual transformer with a hierarchical label GNN. This notebook reads actual public-data preparations and committed aggregate results. Run the README setup and reproduction workflow first; no benchmark values are invented here.


In [ ]:
from pathlib import Path
import json
import os

root = Path.cwd()
if not (root / "semeval").is_dir() and (root.parent / "semeval").is_dir():
    root = root.parent
os.chdir(root)
assert (root / "semeval").is_dir(), "Run from the repository root or experiments directory"


## Data and hierarchy

ProBel preserves its native persuasion-technique hierarchy. CARDS uses its native climate-claim hierarchy. These are public alternatives, not official Task 10 benchmark annotations.


In [ ]:
taxonomy_path = Path("data/public/probel/taxonomy.json")
if taxonomy_path.exists():
    taxonomy = json.loads(taxonomy_path.read_text(encoding="utf-8"))
    print({"fine_labels": len(taxonomy), "parent_categories": len({v["parent"] for v in taxonomy.values()})})
else:
    print("Prepare data with: python -m experiments.reproduce --stage prepare")


## A reproduced core model

Load only your own project checkpoint. The example below performs inference; it does not tune on the held-out record. Outputs stay in the local notebook.


In [ ]:
checkpoint = Path("artifacts/extensions-probel/graph-none-42.joblib")
if checkpoint.exists():
    import joblib
    from semeval.data import read_jsonl
    from semeval.classify import predict
    model = joblib.load(checkpoint)
    example = read_jsonl("data/public/probel/test.jsonl")[0]
    result = predict(model, [example])[0]
    print({"language": result["language"], "predicted_techniques": result["labels"], "parent_categories": result["narratives"]})
else:
    print("Reproduce the classification stage first.")


## Measured experiments

The catalog distinguishes classification scores, explanation format statistics, model-based grounding diagnostics, and processing timings. Compare results within the same study and metric.


In [ ]:
catalog = json.loads(Path("results/experiments.json").read_text(encoding="utf-8"))
core = [row for row in catalog["experiments"] if row.get("role") == "Core"]
for row in core:
    print(row["study"], row["name"], row["metric"], row.get("value"))


## Interactive workspace

Run `python -m semeval.web` in a terminal and open http://127.0.0.1:8765. The website explains the pipeline and includes all experiments in one tab. Analysis uses registered local GNN checkpoints. Grounding diagnostics are model-based, not human validation.
